<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/building-agents/notebooks/m3_loops_reliability.ipynb)
[![Course](https://img.shields.io/badge/Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/building-agents#module-3)

</div>

# Module 3 — Loops & Reliability

**Level:** Beginner | **Time:** ~45 min | [Full reading →](https://rajeevraibhatia.com/curriculum/building-agents#module-3)

### Key concepts
- Add `max_steps` — agents can loop forever without it
- Circuit breaker: same tool call 3× in a row → stop
- Error wrapping: always return errors as strings, never raise
- Retry-with-backoff for rate limits and network errors

In [ ]:
!pip install openai --quiet

In [ ]:
import json
import math
import os
import time
from openai import OpenAI
import openai

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MAX_STEPS = 10

TOOLS = [{"type": "function", "function": {
    "name": "calculator",
    "description": "Evaluate a math expression.",
    "parameters": {"type": "object",
        "properties": {"expression": {"type": "string"}}, "required": ["expression"]}
}}]

TOOL_FNS = {
    "calculator": lambda args: str(eval(args["expression"], {"__builtins__": {}}, vars(math)))
}

def safe_run_tool(name: str, args: dict) -> str:
    """Always returns string — never raises."""
    if name not in TOOL_FNS:
        return f"Error: tool '{name}' not found. Available: {list(TOOL_FNS.keys())}"
    try:
        return TOOL_FNS[name](args)
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

def run_agent(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    recent_calls: list[str] = []
    for step in range(MAX_STEPS):
        r = client.chat.completions.create(model="gpt-4o", messages=messages, tools=TOOLS)
        msg = r.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            call_key = f"{tc.function.name}:{tc.function.arguments}"
            recent_calls.append(call_key)
            if recent_calls[-3:] == [call_key] * 3:
                return "Stopped: agent looping on same tool call."
            result = safe_run_tool(tc.function.name, json.loads(tc.function.arguments))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return f"Stopped: {MAX_STEPS} steps reached."

print(run_agent("What is 15% of 847?"))
print(run_agent("What is the square root of -1?"))  # error case

## Exercise

Wrap the API call with exponential backoff.

In [ ]:
# Exercise: add retry-with-exponential-backoff around the API call
# Retry on openai.RateLimitError and openai.APIConnectionError
# Wait 2s → 4s → 8s between attempts (exponential backoff)
# After 3 failures, re-raise the original error

def chat_with_retry(messages, tools, max_retries=3):
    # Your code here:
    raise NotImplementedError